In [ ]:
import os
from pathlib import Path

import polars as pl
import polars_st as st
import datetime as _dt
import heapq
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import (
    CubicSpline,
    PchipInterpolator,
    Akima1DInterpolator,
    BPoly,
    InterpolatedUnivariateSpline,
)
from scipy.optimize import brentq
import polars.selectors as cs


In [ ]:
db_uri = os.environ["DATABASE_URL"]

In [ ]:
# distances_data["1:3"]

In [ ]:
# df_test_trip = pl.read_json("data/insane_trip.json")

In [ ]:
# df_test_trip.explode("path", "timestamps").with_columns(
#     geometry=st.point("path")
# ).select(~cs.list()).st.explore()

In [ ]:
# df_test_trip.explode("path", "timestamps").unnest("timestamps").sort("parsedValue")

In [ ]:
# Realtime ingestion config and cache toggle.
#
# Set this to True to reuse parquet snapshots.
# Set this to False to pull fresh data from Postgres and re-save parquet snapshots.
READ_FROM_PARQUET = True
WRITE_PARQUET_ON_DB_READ = True

ROUTES = [
    "L",
    "7",
    "A",
    "C",
    "4",
    # "E",
]
DIRECTION = 3  # MTA subway: 1 = southbound, 3 = northbound
WINDOW_DAYS = 50

PARQUET_DIR = Path("data") / "continuous_trajectories"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

RT_TRIP_PARQUET = PARQUET_DIR / "realtime_trip_raw.parquet"
RT_STOP_TIME_PARQUET = PARQUET_DIR / "realtime_stop_time_raw.parquet"

routes_literal = ",".join(f"'{r}'" for r in ROUTES)

if READ_FROM_PARQUET:
    missing = [p for p in [RT_TRIP_PARQUET, RT_STOP_TIME_PARQUET] if not p.exists()]
    if missing:
        missing_str = "\n".join(f"  - {p}" for p in missing)
        raise FileNotFoundError(
            "READ_FROM_PARQUET is True, but snapshot files are missing:\n"
            f"{missing_str}\n"
            "Set READ_FROM_PARQUET = False to pull from Postgres and create them."
        )

    df_rt_trip_raw = pl.read_parquet(RT_TRIP_PARQUET)
    df_rt_stop_time_raw = pl.read_parquet(RT_STOP_TIME_PARQUET)
    realtime_source = "parquet"
else:
    routes_literal = ",".join(f"'{r}'" for r in ROUTES)

    query_rt_trip = f"""
    SELECT id::text AS trip_id,
           source,
           route_id,
           direction,
           created_at,
           shape_ids,
           (data->'consist'->>'car_count')::int AS car_count,
           (data->'consist'->>'car_length_feet')::int AS car_length_feet
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    """

    query_rt_stop_time = f"""
    WITH filtered_trips AS (
        SELECT id
        FROM realtime.trip
        WHERE source = 'mta_subway'
          AND route_id IN ({routes_literal})
          AND direction = {DIRECTION}
          AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    )
    SELECT st.trip_id::text AS trip_id,
           st.stop_id,
           st.arrival,
           st.data->'platform_edges' AS platform_edges
    FROM realtime.stop_time st
    JOIN filtered_trips ft ON ft.id = st.trip_id
    ORDER BY st.trip_id, st.arrival
    """

    df_rt_trip_raw = pl.read_database_uri(query_rt_trip, db_uri)
    df_rt_stop_time_raw = pl.read_database_uri(query_rt_stop_time, db_uri)
    realtime_source = "database"

    if WRITE_PARQUET_ON_DB_READ:
        df_rt_trip_raw.write_parquet(RT_TRIP_PARQUET)
        df_rt_stop_time_raw.write_parquet(RT_STOP_TIME_PARQUET)

# df_rt = (
#     df_rt_stop_time_raw.join(df_rt_trip_raw, on="trip_id", how="inner")
#     .select(
#         [
#             "trip_id",
#             "route_id",
#             "direction",
#             "created_at",
#             "shape_ids",
#             "car_count",
#             "stop_id",
#             "arrival",
#         ]
#     )
#     .sort(["trip_id", "arrival"])
# )

print(f"Realtime source: {realtime_source}")
print(
    f"Loaded {df_rt_stop_time_raw.height} stop-time rows across "
    f"{df_rt_trip_raw['trip_id'].n_unique()} trips "
    f"on routes {ROUTES} direction={DIRECTION} over the last {WINDOW_DAYS} days"
)
# df_rt_stop_time_raw.head()


In [ ]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) AS geometry,
       data->'platform_edges' AS platform_edges_data
FROM static.stop
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "stops.parquet")
df_stops = pl.read_parquet(PARQUET_DIR / "stops.parquet")


In [ ]:
query_shapes = f"""
WITH unique_shape_arrays AS (
    SELECT DISTINCT shape_ids
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
),
unnested AS (
    SELECT shape_ids,
           x.shape_id,
           x.ord
    FROM unique_shape_arrays,
         UNNEST(shape_ids) WITH ORDINALITY AS x(shape_id, ord)
),
joined AS (
    SELECT u.shape_ids,
           u.ord,
           s.geom
    FROM unnested u
    JOIN static.shape s ON s.id = u.shape_id AND s.source = 'mta_subway'
),
merged AS (
    SELECT shape_ids,
           ST_AsEWKB(ST_MakeLine(geom ORDER BY ord)) AS geometry
    FROM joined
    GROUP BY shape_ids
)
SELECT shape_ids,
       ST_Length(geometry::geography) as trip_length_m,
       geometry
FROM merged
"""
if READ_FROM_PARQUET:
    df_trip_shapes = pl.read_parquet(PARQUET_DIR / "trip_shapes.parquet")
else:
    df_trip_shapes = pl.read_database_uri(query_shapes, db_uri)
    if WRITE_PARQUET_ON_DB_READ:
        df_trip_shapes.write_parquet(PARQUET_DIR / "trip_shapes.parquet")


In [ ]:
# df_segments.filter(
#     (pl.col("route_id") == "1") & (pl.col("stop_id").is_in(["126", "127"]))
# ).select(~cs.binary())
# df_segments.filter((pl.col("route_id") == "1")).select(~cs.binary())[20:30]

In [ ]:
# print(f"Realtime source in use: {realtime_source}")
# print(
#     f"Trip rows: {df_rt_trip_raw.height:,} | "
#     f"Stop-time rows: {df_rt_stop_time_raw.height:,} | "
#     f"Joined rows: {df_trips_with_shapes.height:,}"
# )
# print(
#     f"Trips: {df_trips_with_shapes['trip_id'].n_unique():,} | "
#     f"Routes: {df_trips_with_shapes['route_id'].n_unique():,} | "
#     f"Window: {WINDOW_DAYS} day(s)"
# )
# df_rt.head()

In [ ]:
df_trips_with_shapes = df_rt_trip_raw.join(
    df_trip_shapes, on="shape_ids", how="left"
).with_columns(
    # TODO: decide to use this consist length or the one from the platform edge static data
    consist_length_m=pl.col("car_length_feet") * pl.col("car_count") * 0.3048
)

In [ ]:
df_platform_edges = (
    df_stops.with_columns(
        pl.col("platform_edges_data")
        .str.json_decode(
            dtype=pl.List(
                pl.Struct(
                    {
                        "id": pl.String,
                        "length_ft": pl.Float64,
                        "car_markers": pl.List(
                            pl.Struct(
                                {
                                    "marked_as": pl.String,
                                    "consist_length_ft": pl.Float64,
                                    "position_ft": pl.Float64,
                                    "direction": pl.String,
                                }
                            )
                        ),
                    }
                )
            )
        )
        .name.keep()
    )
    .explode("platform_edges_data")
    .rename({"id": "stop_id"})
    .unnest("platform_edges_data")
    .rename({"id": "platform_edge_id", "length_ft": "platform_edge_length_ft"})
    .explode("car_markers")
    .unnest("car_markers")
    .with_columns(
        platform_edge_length_m=pl.col("platform_edge_length_ft") * 0.3048,
        position_m=pl.col("position_ft") * 0.3048,
        consist_length_m=pl.col("consist_length_ft") * 0.3048,
        # TODO: maybe add east/west if we add to rust struct
        direction=pl.col("direction").replace_strict({"NORTH": 1, "SOUTH": 3}),
    )
    .drop("platform_edge_length_ft", "position_ft", "consist_length_ft")
)
# df_platform_edges

In [ ]:
# Some stop times have multiple platform edges associated with them.
# Keep exact car-count/marked_as matches when possible, then fall back to the
# platform edge whose direction-adjusted position is closest to the consist length.
#
# platform_edge position_m is measured relative to railway north, so we flip it
# for southbound trips before comparing against the consist length.
df_platform_edges_match = df_platform_edges.rename(
    {
        "marked_as": "edge_marked_as",
        "direction": "platform_edge_direction",
        "consist_length_m": "edge_consist_length_m",
    }
)

df_rt_stop_time_platform_edges = (
    df_rt_stop_time_raw.with_row_index("stop_time_row_id")
    .with_columns(pl.col("platform_edges").str.json_decode(dtype=pl.List(pl.String)))
    .explode("platform_edges")
    .rename({"platform_edges": "platform_edge_id"})
    .join(
        df_trips_with_shapes.select(
            "trip_id",
            "route_id",
            "direction",
            pl.col("car_count").cast(pl.String).alias("marked_as"),
            "consist_length_m",
        ),
        on="trip_id",
        how="left",
    )
    .join(df_platform_edges_match, on="platform_edge_id", how="left")
    .with_columns(
        position_from_trip_direction_m=pl.when(pl.col("direction") == 3)
        .then(pl.col("position_m"))
        .otherwise(pl.col("platform_edge_length_m") - pl.col("position_m")),
        match_is_exact=(pl.col("marked_as") == pl.col("edge_marked_as"))
        & (pl.col("direction") == pl.col("platform_edge_direction")),
    )
    .with_columns(
        position_delta_m=(
            pl.col("position_from_trip_direction_m") - pl.col("consist_length_m")
        ).abs(),
        match_rank=pl.when(pl.col("match_is_exact")).then(0).otherwise(1),
        match_method=pl.when(pl.col("match_is_exact"))
        .then(pl.lit("exact"))
        .otherwise(pl.lit("nearest_position")),
    )
    .sort(["stop_time_row_id", "match_rank", "position_delta_m", "platform_edge_id"])
    .group_by("stop_time_row_id", maintain_order=True)
    .first()
    .drop("stop_time_row_id", "stop_id_right")
)

# df_rt_stop_time_platform_edges

In [ ]:
df_stop_times_w_projected_stops = (
    df_rt_stop_time_platform_edges.filter(st.geom().is_not_null())
    .join(df_trips_with_shapes.rename({"geometry": "trip_geometry"}), on="trip_id")
    # some trips are bugged and don't have shapes, so we have to filter them out
    .filter(st.geom("trip_geometry").is_not_null())
    # .with_columns(
    #     cs.binary().st.to_srid(6538)
    # )  # convert to projected CRS that uses meters as units
    .with_columns(
        cum_distance_center=st.geom("trip_geometry").st.project(
            st.geom(), normalized=True
        )
    )
)

In [ ]:
df_stop_times_w_spatial_window = (
    df_stop_times_w_projected_stops.with_columns(
        S_centroid=(pl.col("cum_distance_center") * pl.col("trip_length_m")),
    )
    .with_columns(
        # TODO: maybe increase position_m and platform_edge_length_m
        # Fallback to centroid and generic dwell if platform data is missing
        platform_edge_length_m=pl.col("platform_edge_length_m").fill_null(10.0),
        position_m=pl.col("position_m").fill_null(5.0),
        consist_length_m=pl.col("consist_length_m").fill_null(150.0),
    )
    .with_columns(
        # Clamp to 0.0 to prevent negative space at terminal stations
        S_start=pl.max_horizontal(
            0.0, pl.col("S_centroid") - (pl.col("platform_edge_length_m") / 2.0)
        ),
        S_end=pl.min_horizontal(
            pl.col("trip_length_m"),
            pl.col("S_centroid") + (pl.col("platform_edge_length_m") / 2.0),
        ),
    )
    .with_columns(
        # This is the point along the track where the train should be stopped at
        S_mark=(pl.col("S_start") + pl.col("position_m")),
    )
    .with_columns(
        # Tail clears when the nose travels the rest of the platform + the train length
        S_tail_clear=pl.min_horizontal(
            pl.col("trip_length_m")
            + pl.col("consist_length_m"),  # Allow it to overhang the end of the trip
            pl.col("S_end") + pl.col("consist_length_m"),
        )
    )
)

In [ ]:
# Typical MTA service rates
A_DECEL = 1.0  # m/s^2 (braking)
A_ACCEL = 0.8  # m/s^2 (accelerating)
DWELL_SECONDS = 30

df_kinematic = (
    df_stop_times_w_spatial_window.with_columns(
        # 1. Calculate the spatial distance required for entry and exit
        # d_entry=pl.max_horizontal(0.0, pl.col("S_mark") - pl.col("S_start")),
        # d_exit=pl.max_horizontal(0.0, pl.col("S_tail_clear") - pl.col("S_mark")),
        d_entry=pl.col("S_mark") - pl.col("S_start"),
        d_exit=pl.col("S_tail_clear") - pl.col("S_mark"),
    )
    .with_columns(
        # 2. Kinematic time = sqrt(2d / a)
        dt_decel_s=(2.0 * pl.col("d_entry") / A_DECEL).sqrt(),
        dt_accel_s=(2.0 * pl.col("d_exit") / A_ACCEL).sqrt(),
    )
    .with_columns(
        # 3. Establish the core dwell window (Knots 2 & 3)
        t_2=pl.col("arrival"),
        t_3=pl.col("arrival") + pl.duration(seconds=DWELL_SECONDS),
    )
    .with_columns(
        # 4. Project the entry (Knot 1) and exit (Knot 4) timestamps
        # We multiply by 1e6 to convert float seconds to microseconds for Polars duration math
        t_1=pl.col("t_2")
        - pl.duration(microseconds=(pl.col("dt_decel_s") * 1e6).cast(pl.Int64)),
        t_4=pl.col("t_3")
        + pl.duration(microseconds=(pl.col("dt_accel_s") * 1e6).cast(pl.Int64)),
    )
)

In [ ]:
df_exploded = (
    df_kinematic.with_columns(
        # Bundle the 4 events into a list of structs
        # t_event is the timestamp of the event, s_m is the spatial position along the track, and v_clamp is the speed limit at that knot (None means no change, 0.0 means stop)
        pl.concat_list(
            [
                # Knot 1: start of deceleration (speed clamp may be added later based on approach)
                pl.struct(
                    [
                        pl.col("t_1").alias("t_event"),
                        pl.col("S_start").alias("s_m"),
                        pl.lit(None).cast(pl.Float64).alias("v_clamp"),
                    ]
                ),
                # Knots 2 & 3: dwell window (train should be stopped, so v_clamp=0.0)
                pl.struct(
                    [
                        pl.col("t_2").alias("t_event"),
                        pl.col("S_mark").alias("s_m"),
                        pl.lit(0.0).alias("v_clamp"),
                    ]
                ),
                pl.struct(
                    [
                        pl.col("t_3").alias("t_event"),
                        pl.col("S_mark").alias("s_m"),
                        pl.lit(0.0).alias("v_clamp"),
                    ]
                ),
                # Knot 4: end of dwell/start of acceleration (speed clamp may be added later based on departure)
                pl.struct(
                    [
                        pl.col("t_4").alias("t_event"),
                        pl.col("S_tail_clear").alias("s_m"),
                        pl.lit(None).cast(pl.Float64).alias("v_clamp"),
                    ]
                ),
            ]
        ).alias("kinematic_knots")
    )
    .explode("kinematic_knots")
    .unnest("kinematic_knots")
)

# Sort to ensure strict time-series integrity for the interpolator
df_final_knots = df_exploded.sort(["trip_id", "t_event"])
BACKTRACK_TOL_M = 0.5


def _collapse_backtracking_knots(
    trip_knots: pl.DataFrame, backtrack_tol_m: float = BACKTRACK_TOL_M
):
    """Drop knots that do not advance by more than the backtrack tolerance."""
    if trip_knots.height == 0:
        return trip_knots, 0

    keep_rows = [0]
    collapsed = 0
    s_values = trip_knots["s_m"].to_list()

    for i in range(1, len(s_values)):
        if s_values[i] <= s_values[keep_rows[-1]] + backtrack_tol_m:
            collapsed += 1
            continue
        keep_rows.append(i)

    return trip_knots[keep_rows], collapsed


collapsed_trip_knots = []
total_collapsed = 0
trips_with_collapsed = 0

for trip_knots in df_final_knots.partition_by(
    ["trip_id", "route_id", "direction"], maintain_order=True
):
    trip_knots = trip_knots.sort("t_event")
    filtered_knots, collapsed = _collapse_backtracking_knots(trip_knots)

    if collapsed:
        trip_meta = trip_knots.select("trip_id", "route_id", "direction").row(
            0, named=True
        )
        total_collapsed += collapsed
        trips_with_collapsed += 1
        print(
            f"[knot-collapse] trip_id={trip_meta['trip_id']} route_id={trip_meta['route_id']} ",
            f"direction={trip_meta['direction']} collapsed {collapsed} knot(s) ",
            f"(tol={BACKTRACK_TOL_M:.2f}m)",
        )

    collapsed_trip_knots.append(filtered_knots)

df_final_knots = pl.concat(collapsed_trip_knots).sort(["trip_id", "t_event"])

if total_collapsed:
    print(
        f"[knot-collapse] collapsed {total_collapsed} knot(s) across {trips_with_collapsed} trip(s)"
    )
else:
    print(f"[knot-collapse] no backwards knots collapsed (tol={BACKTRACK_TOL_M:.2f}m)")

In [ ]:
df_final_knots

In [ ]:
trip_ids_with_more_than_3_stop_times = (
    df_rt_stop_time_raw.group_by("trip_id")
    .len()
    .filter(pl.col("len") > 3)
    .select("trip_id")
    .sample(1)
)


# df_trip = df_final_knots.join(trip_ids_with_more_than_3_stop_times, on="trip_id")
df_trip = df_final_knots.filter(trip_id="019defc4-2443-7003-96e7-691667701584")
test_trip = df_trip["trip_id"].to_list()[0]
df_final_knots.filter(pl.col("trip_id") == test_trip).select(~cs.binary()).sort(
    "arrival"
).head(5)

In [ ]:
# TODO: remove these since we have the other general function that takes interpolator type as arg
def build_trip_interpolator(trip_df):
    t_start = trip_df["t_event"][0]
    t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])

    s_vals = trip_df["s_m"].to_numpy()
    v_clamps = trip_df["v_clamp"].to_numpy()

    y_derivatives = []
    for s, v in zip(s_vals, v_clamps):
        if np.isnan(v):
            y_derivatives.append([s])  # Let Scipy calculate the velocity
        else:
            y_derivatives.append([s, v])  # Clamp the velocity to 'v' (e.g., 0.0)

    # Create the BPoly curve
    interp = BPoly.from_derivatives(t_sec, y_derivatives)
    return (
        interp,
        t_start,
        t_sec[-1],
    )  # Return the function, the start time, and total duration


def build_trip_interpolator_pchip(trip_df):
    t_start = trip_df["t_event"][0]
    t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
    s_vals = trip_df["s_m"].to_numpy()

    # PCHIP automatically handles the flat dwell and ensures continuous velocity
    interp = PchipInterpolator(t_sec, s_vals, extrapolate=False)

    return interp, t_start, t_sec[-1]

In [ ]:
def plot_station_kinematics(interp, t_start, t_end, station_name):
    """
    interp: The BPoly interpolator object
    t_start, t_end: Time range in seconds around the station stop
    """
    # Generate dense time array for smooth plotting
    ts = np.linspace(t_start, t_end, 500)

    # Extract derivatives directly from the BPoly object
    s_vals = interp(ts)
    v_vals = interp.derivative(1)(ts)
    a_vals = interp.derivative(2)(ts)

    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    # Position
    axs[0].plot(ts, s_vals, label="Interpolated Position", color="blue")
    axs[0].set_ylabel("Distance (m)")
    axs[0].set_title(f"Kinematic Profiles: {station_name} Stop")
    axs[0].grid(alpha=0.3)

    # Velocity
    axs[1].plot(ts, v_vals, label="Velocity", color="green")
    axs[1].axhline(0, color="black", linestyle="--", alpha=0.5)
    axs[1].set_ylabel("Velocity (m/s)")
    axs[1].grid(alpha=0.3)

    # Acceleration
    axs[2].plot(ts, a_vals, label="Acceleration", color="red")
    axs[2].axhline(1.0, color="gray", linestyle=":", label="Service Max (1.0 m/s^2)")
    axs[2].axhline(-1.0, color="gray", linestyle=":", label="Braking Max (-1.0 m/s^2)")
    axs[2].set_ylabel("Acceleration (m/s^2)")
    axs[2].set_xlabel("Time (seconds)")
    axs[2].legend()
    axs[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_trip_trajectory(interp, t_max, raw_stop_times, raw_stop_distances):
    ts = np.linspace(0, t_max, 1000)
    s_vals = interp(ts)

    plt.figure(figsize=(12, 6))

    # The smooth predicted trajectory
    plt.plot(ts, s_vals, label="BPoly Interpolated Path", color="darkblue", linewidth=2)

    # The raw data points (the original 'jumping' data)
    plt.scatter(
        raw_stop_times,
        raw_stop_distances,
        color="orange",
        zorder=5,
        label="MTA API Stop Predictions",
    )

    plt.title("Continuous Trip Trajectory vs. Discrete API Data")
    plt.xlabel("Time since trip start (seconds)")
    plt.ylabel("Distance along route (meters)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
def plot_error_distribution(abs_errors_seconds):
    plt.figure(figsize=(10, 5))

    # Histogram of errors
    plt.hist(
        abs_errors_seconds, bins=50, color="steelblue", edgecolor="black", alpha=0.7
    )

    # Add vertical lines for Median and 90th Percentile
    median_err = np.median(abs_errors_seconds)
    p90_err = np.percentile(abs_errors_seconds, 90)

    plt.axvline(
        median_err,
        color="red",
        linestyle="dashed",
        linewidth=2,
        label=f"Median Error: {median_err:.1f}s",
    )
    plt.axvline(
        p90_err,
        color="orange",
        linestyle="dashed",
        linewidth=2,
        label=f"90th Percentile: {p90_err:.1f}s",
    )

    plt.title("Cross-Validation: Interpolation Prediction Error")
    plt.xlabel("Absolute Error Margin (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
# To get the position at any time:
# current_s = interp((current_time - t_start).total_seconds())
interp_function, trip_start_time, trip_duration = build_trip_interpolator(df_trip)
interp_function_p, trip_start_time_p, trip_duration_p = build_trip_interpolator_pchip(
    df_trip
)

In [ ]:
# Generate an array of seconds from 0 to the end of the trip
# (This is how you would call it to generate GeoJSON points for Deck.gl!)
time_array = np.linspace(0, trip_duration, 500)

# 4a. Get Position (Distance along the track)
positions_m = interp_function(time_array)

# 4b. Get Velocity (1st Derivative)
velocities_m_s = interp_function.derivative(1)(time_array)

# 4c. Get Acceleration (2nd Derivative)
accelerations_m_s2 = interp_function.derivative(2)(time_array)

In [ ]:
# 1. Identify the time window around a specific station.
# For the first station, the approach starts at t=0. Let's look at the first 60 seconds.
t_window_start = -5.0  # Look slightly before the spline starts
t_window_end = 120.0  # Look until after it accelerates out

plot_station_kinematics(
    interp=interp_function,
    t_start=t_window_start,
    t_end=t_window_end,
    station_name="Station A (Initial Departure)",
)

In [ ]:
plot_station_kinematics(
    interp=interp_function_p,
    t_start=t_window_start,
    t_end=t_window_end,
    station_name="Station A (Initial Departure)",
)

In [ ]:
# Assuming plot_trip_trajectory is defined in a previous cell

# 1. Extract the discrete points from your dataframe
# Convert the Polars datetime column into relative seconds from the trip start
raw_times_seconds = [(t - trip_start_time).total_seconds() for t in df_trip["t_event"]]
raw_distances_meters = df_trip["s_m"].to_numpy()

# 2. Call the function
plot_trip_trajectory(
    interp=interp_function,
    t_max=trip_duration,
    raw_stop_times=raw_times_seconds,
    raw_stop_distances=raw_distances_meters,
)

In [ ]:
import json
import numpy as np
import shapely.wkb


def export_trip_to_deckgl(df_shapes, trip_id, interp_func, duration):
    # 1. Get the geometry and total length for this specific trip
    trip_row = df_shapes.filter(pl.col("trip_id") == trip_id).row(0, named=True)

    # Load the EWKB byte array into a Shapely LineString
    trip_geom = shapely.wkb.loads(trip_row["geometry"])
    trip_length_m = trip_row["trip_length_m"]

    # 2. Generate a time array (1-second intervals for 60fps animation)
    time_array = np.linspace(0, duration, int(duration) + 1)

    # 3. Get the predicted track distances (s) at each second
    distances_m = interp_func(time_array)

    path_coords = []
    timestamps = []

    for t, s in zip(time_array, distances_m):
        # Clamp distance to avoid bounds errors (SciPy curves can sometimes overshoot minutely)
        s_clamped = max(0.0, min(s, trip_length_m))

        # Normalize distance [0.0 - 1.0] to interpolate along the Shapely geometry
        normalized_s = s_clamped / trip_length_m
        pt = trip_geom.interpolate(normalized_s, normalized=True)

        # Deck.gl expects [longitude, latitude]
        path_coords.append([pt.x, pt.y])
        timestamps.append(float(t))

    # 4. Format for the Deck.gl TripsLayer
    export_data = [{"trip_id": trip_id, "path": path_coords, "timestamps": timestamps}]

    # 5. Save to JSON
    filename = f"deckgl_trip_{trip_id}.json"
    with open(filename, "w") as f:
        json.dump(export_data, f)

    print(f"Exported {filename} successfully! ({len(timestamps)} frames)")


# Run the export function
export_trip_to_deckgl(
    df_shapes=df_trips_with_shapes,
    trip_id=test_trip,
    interp_func=interp_function,  # Or interp_function_p for PCHIP
    duration=trip_duration,
)

In [ ]:
def debug_trip_knot_order(trip_id, df_knots, df_stops_raw=None, df_stop_times_raw=None):
    """
    Check a trip for non-monotone knots and print violations with stop details.
    """
    trip_knots = df_knots.filter(pl.col("trip_id") == trip_id).sort("t_event")

    if trip_knots.height == 0:
        print(f"Trip {trip_id} not found")
        return

    print(f"\n=== Trip {trip_id} Knot Order Debug ===")
    print(f"Total knots: {trip_knots.height}\n")

    # Join with stop info if available
    if df_stops_raw is not None and df_stop_times_raw is not None:
        # Get stop IDs for this trip
        trip_stops = (
            df_stop_times_raw.filter(pl.col("trip_id") == trip_id)
            .select("stop_id", "arrival")
            .join(df_stops_raw.select("id", "name"), left_on="stop_id", right_on="id")
        )

        # Create a mapping from arrival time to stop info
        stop_map = {}
        for row in trip_stops.rows(named=True):
            stop_map[row["arrival"]] = {
                "stop_id": row["stop_id"],
                "stop_name": row["name"],
            }
    else:
        stop_map = {}

    # Extract relevant columns
    times = trip_knots["t_event"].to_list()
    distances = trip_knots["s_m"].to_numpy()
    stop_ids = (
        trip_knots["stop_id"].to_list()
        if "stop_id" in trip_knots.columns
        else [None] * len(times)
    )

    print("Knot sequence:")
    for i, (t, s, stop_id) in enumerate(zip(times, distances, stop_ids)):
        # Try to find stop info
        stop_info = stop_map.get(t, {})
        stop_detail = ""
        if stop_info:
            stop_detail = (
                f" | Stop: {stop_info['stop_name']} (ID: {stop_info['stop_id']})"
            )
        elif stop_id:
            stop_detail = f" | Stop ID: {stop_id}"

        print(f"  Knot {i}: t={t}, s={s:.2f}m{stop_detail}")

    # Check time monotonicity
    time_violations = []
    for i in range(1, len(times)):
        if times[i] <= times[i - 1]:
            time_violations.append((i - 1, i, times[i - 1], times[i]))

    if time_violations:
        print("\n⚠️  TIME VIOLATIONS (not strictly increasing):")
        for idx1, idx2, t1, t2 in time_violations:
            print(f"  Knot {idx1} → {idx2}: {t1} → {t2}")
    else:
        print("\n✓ Times are strictly increasing")

    # Check distance monotonicity (should generally increase)
    distance_violations = []
    for i in range(1, len(distances)):
        if distances[i] < distances[i - 1]:
            distance_violations.append((i - 1, i, distances[i - 1], distances[i]))

    if distance_violations:
        print("\n⚠️  DISTANCE VIOLATIONS (not monotone increasing):")
        for idx1, idx2, s1, s2 in distance_violations:
            print(f"  Knot {idx1} → {idx2}: {s1:.2f}m → {s2:.2f}m (Δ={s2 - s1:.2f}m)")
    else:
        print("\n✓ Distances are monotone increasing")


# Example usage:
# debug_trip_knot_order(test_trip, df_final_knots, df_stops, df_rt_stop_time_raw)


In [ ]:
def analyze_invalid_trips(df_knots, df_stop_times_raw):
    """
    Analyze all trips and report statistics about invalid/problematic trips.
    Checks for:
    - Non-monotone times (not strictly increasing)
    - Non-monotone distances (going backwards)
    - Stop times that aren't monotone increasing
    """
    print("=== Invalid Trip Analysis ===\n")

    # Get unique trip IDs
    trip_ids = df_knots["trip_id"].unique().to_list()
    print(f"Total unique trips in knots: {len(trip_ids)}\n")

    invalid_trips = {
        "non_monotone_time": [],
        "non_monotone_distance": [],
        "empty_trips": [],
    }

    # Check each trip
    for trip_id in trip_ids:
        trip_knots = df_knots.filter(pl.col("trip_id") == trip_id).sort("t_event")

        if trip_knots.height == 0:
            invalid_trips["empty_trips"].append(trip_id)
            continue

        times = trip_knots["t_event"].to_list()
        distances = trip_knots["s_m"].to_numpy()

        # Check time monotonicity
        has_time_violation = any(times[i] <= times[i - 1] for i in range(1, len(times)))
        if has_time_violation:
            invalid_trips["non_monotone_time"].append(trip_id)

        # Check distance monotonicity
        has_distance_violation = any(
            distances[i] < distances[i - 1] for i in range(1, len(distances))
        )
        if has_distance_violation:
            invalid_trips["non_monotone_distance"].append(trip_id)

    # Check stop times for non-monotone arrivals
    print("Stop time monotonicity analysis:")
    stop_time_violations = []

    for trip_id in trip_ids:
        trip_stops = df_stop_times_raw.filter(pl.col("trip_id") == trip_id).sort(
            "arrival"
        )
        arrivals = trip_stops["arrival"].to_list()

        for i in range(1, len(arrivals)):
            if arrivals[i] <= arrivals[i - 1]:
                stop_time_violations.append(trip_id)
                break

    print(f"  Trips with non-monotone stop times: {len(stop_time_violations)}")
    if stop_time_violations:
        print(f"    Example trip IDs: {stop_time_violations[:5]}\n")

    # Print summary
    print("Knot-based violations:")
    print(f"  Non-monotone times: {len(invalid_trips['non_monotone_time'])} trips")
    if invalid_trips["non_monotone_time"]:
        print(f"    Examples: {invalid_trips['non_monotone_time'][:5]}")

    print(
        f"  Non-monotone distances: {len(invalid_trips['non_monotone_distance'])} trips"
    )
    if invalid_trips["non_monotone_distance"]:
        print(f"    Examples: {invalid_trips['non_monotone_distance'][:5]}")

    print(f"  Empty trips: {len(invalid_trips['empty_trips'])}")

    # Summary statistics
    valid_trips = len(trip_ids) - len(
        set(
            invalid_trips["non_monotone_time"]
            + invalid_trips["non_monotone_distance"]
            + invalid_trips["empty_trips"]
        )
    )

    print(
        f"\n✓ Valid trips: {valid_trips} / {len(trip_ids)} ({100 * valid_trips / len(trip_ids):.1f}%)"
    )

    return invalid_trips


# Example usage:
# invalid_stats = analyze_invalid_trips(df_final_knots, df_rt_stop_time_raw)


In [ ]:
invalid_stats = analyze_invalid_trips(df_final_knots, df_rt_stop_time_raw)

In [ ]:
# debug_trip_knot_order(test_trip, df_final_knots, df_stops, df_rt_stop_time_raw)

In [ ]:
# increase max rows
pl.Config.set_tbl_rows(100)

# df_stop_times_w_spatial_window.filter(
#     trip_id="019deffc-269d-7760-bbdb-366bb7c83ad9"
# ).sort("arrival")
# df_stop_times_w_spatial_window.filter(
#     trip_id="019df0a2-1aed-7f52-9288-875f9ceb2a3c"
# ).sort("arrival")

In [ ]:
VALIDATION_INTERPOLATION_METHODS = (
    "pchip",
    "cubic_clamped",
    "cubic_natural",
    "cubic",
    "linear",
    "bpoly",
)


def _build_validation_interpolator(t_sec, s_vals, interp_type="pchip", v_clamps=None):
    method = interp_type.lower()
    t_sec = np.asarray(t_sec, dtype=float)
    s_vals = np.asarray(s_vals, dtype=float)

    if method == "bpoly":
        if v_clamps is None:
            v_clamps = np.full(len(s_vals), np.nan)
        v_clamps = np.asarray(v_clamps, dtype=float)
        y_derivatives = []
        for s, v in zip(s_vals, v_clamps):
            if np.isnan(v):
                y_derivatives.append([float(s)])
            else:
                y_derivatives.append([float(s), float(v)])
        return BPoly.from_derivatives(t_sec, y_derivatives)

    if method == "pchip":
        return PchipInterpolator(t_sec, s_vals, extrapolate=False)

    if method == "cubic":
        # not-a-knot cubic spline (default in SciPy)
        return CubicSpline(t_sec, s_vals, extrapolate=False)

    if method == "cubic_clamped":
        return CubicSpline(t_sec, s_vals, bc_type="clamped", extrapolate=False)

    if method == "cubic_natural":
        return CubicSpline(t_sec, s_vals, bc_type="natural", extrapolate=False)

    if method == "linear":
        return InterpolatedUnivariateSpline(t_sec, s_vals, k=1, ext=2)

    raise ValueError(
        f"Unsupported interpolation method: {interp_type}. "
        f"Expected one of {VALIDATION_INTERPOLATION_METHODS}"
    )


def build_validation_interpolator(trip_df, interp_type="pchip"):
    t_start = trip_df["t_event"][0]
    t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
    s_vals = trip_df["s_m"].to_numpy()
    v_clamps = (
        trip_df["v_clamp"].to_numpy()
        if "v_clamp" in trip_df.columns
        else np.full(len(s_vals), np.nan)
    )

    interp = _build_validation_interpolator(t_sec, s_vals, interp_type, v_clamps)
    return interp, t_start, t_sec[-1]


In [ ]:
def leave_one_out_cv_for_trip(trip_df, interp_type="bpoly"):
    """Perform leave-one-out CV for internal stops on a trip.
    Returns a list of absolute errors (seconds).
    trip_df must be sorted by `t_event`.
    interp_type: 'bpoly', 'pchip', 'cubic', or 'linear'
    """
    # convert to numpy arrays for SciPy
    t_start = trip_df["t_event"][0]
    t_sec_full = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
    s_full = trip_df["s_m"].to_numpy()
    v_full = (
        trip_df["v_clamp"].to_numpy()
        if "v_clamp" in trip_df.columns
        else np.full(len(s_full), np.nan)
    )

    n = len(s_full)
    if n < 4:
        return []

    errors = []

    # iterate internal stops only (skip first and last)
    for i in range(1, n - 1):
        # Build training arrays by masking out the i-th point
        mask = np.ones(n, dtype=bool)
        mask[i] = False

        t_train = t_sec_full[mask]
        s_train = s_full[mask]
        v_train = v_full[mask]

        # Build interpolator from training set
        try:
            interp = _build_validation_interpolator(
                t_train, s_train, interp_type=interp_type, v_clamps=v_train
            )
        except Exception:
            continue

        # Bracket search for root: between neighboring training times around removed point
        t_low = t_sec_full[max(0, i - 1)]
        t_high = t_sec_full[min(n - 1, i + 1)]
        s_target = float(s_full[i])

        def f(t):
            return float(interp(t) - s_target)

        try:
            # If bracket values already equal sign, brentq will fail; check monotonicity by sampling
            t_pred = brentq(f, t_low, t_high, maxiter=50)
            t_actual = float(t_sec_full[i])
            errors.append(abs(t_pred - t_actual))
        except Exception:
            # fallback: try a small expanded bracket around the hidden point
            try:
                t_pred = brentq(f, t_low - 1.0, t_high + 1.0, maxiter=50)
                errors.append(abs(t_pred - float(t_sec_full[i])))
            except Exception:
                continue

    return errors


In [ ]:
# -- Run leave-one-out CV across a sample of trips and summarize errors
from math import sqrt


# TODO: allow passing in a list of trip ids so we can do focused CV on known problematic trips and compare interpolation methods head-to-head on the same trips
def run_cv_on_sample(
    df_knots,
    trip_ids=None,
    sample_size=50,
    interp_methods=None,
    random_state=None,
):
    """Run leave-one-out CV across a sample of trips for multiple interpolation methods.

    Args:
        df_knots: DataFrame containing knots for all trips.
        trip_ids: optional list of trip ids to test. If None, a random sample from df_knots is used.
        sample_size: size of random sample when trip_ids is None.
        interp_methods: iterable of interpolation method names (strings). If None, uses VALIDATION_INTERPOLATION_METHODS.
        random_state: optional int for reproducible sampling.

    Returns:
        results: dict mapping interp_method -> {n, mae, rmse, errors(array)}
    """
    if interp_methods is None:
        interp_methods = VALIDATION_INTERPOLATION_METHODS

    # Build list of trip ids
    all_trip_ids = df_knots.select("trip_id").unique().to_series().to_list()
    if trip_ids is None:
        rng = np.random.RandomState(random_state)
        if len(all_trip_ids) > sample_size:
            trip_ids = rng.choice(all_trip_ids, sample_size, replace=False).tolist()
        else:
            trip_ids = all_trip_ids

    results = {}
    # For each interpolation method, collect errors across the same sample of trips
    for method in interp_methods:
        errors = []
        for tid in trip_ids:
            trip_df = df_knots.filter(pl.col("trip_id") == tid).sort("t_event")
            errs = leave_one_out_cv_for_trip(trip_df, interp_type=method)
            errors.extend(errs)

        if not errors:
            results[method] = {
                "n": 0,
                "mae": None,
                "rmse": None,
                "errors": np.array([]),
            }
            continue

        arr = np.array(errors)
        mae = float(arr.mean())
        rmse = float(np.sqrt((arr**2).mean()))
        results[method] = {"n": int(len(arr)), "mae": mae, "rmse": rmse, "errors": arr}

    # Plot comparative boxplot of errors (skip empty methods)
    plt.figure(figsize=(12, 6))
    labels = []
    data = []
    for m in interp_methods:
        d = results[m]["errors"]
        if len(d) > 0:
            labels.append(m)
            data.append(d)

    if data:
        plt.boxplot(data, tick_labels=labels)
        plt.ylabel("Absolute Error (seconds)")
        plt.title("Leave-One-Out CV Error Comparison by Interpolator")
        plt.grid(axis="y", alpha=0.3)
        plt.show()
    else:
        print("No CV errors to plot for the selected methods/sample.")

    # Print concise table
    print("CV summary by method:")
    for m in interp_methods:
        r = results[m]
        if r["n"]:
            print(f" - {m}: n={r['n']}, MAE={r['mae']:.2f}s, RMSE={r['rmse']:.2f}s")
        else:
            print(f" - {m}: no valid CV trials collected")

    return results


# Example usage: compare methods on a random sample
cv_summary = run_cv_on_sample(
    df_final_knots,
    sample_size=100,
    interp_methods=["pchip", "bpoly", "cubic_clamped", "linear"],
    random_state=42,
)
cv_summary

In [ ]:
# # Not as good as normal cubicspline since we were forcing the flat velocity at specific stops.
# cv_summary = run_cv_on_sample(df_final_knots, sample_size=100, interp_type="bpoly")
# cv_summary

In [ ]:
# cv_summary = run_cv_on_sample(
#     df_final_knots, sample_size=100, interp_type="cubic_clamped"
# )
# cv_summary

In [ ]:
# cv_summary = run_cv_on_sample(df_final_knots, sample_size=100, interp_type="linear")
# cv_summary

In [ ]:
def evaluate_precision_degradation(trip_list, interp_methods=None, plot=True):
    """Evaluate precision degradation for multiple trips and interpolation methods.

    Args:
        trip_list: list of per-trip DataFrames (each sorted by t_event). Can be a single trip_df as well.
        interp_methods: iterable of interpolation method names. If None uses VALIDATION_INTERPOLATION_METHODS.
        plot: whether to produce summary plots.

    Returns:
        summary: dict[method] -> dict(dtype -> {mean_v_err, std_v_err, mean_a_err, std_a_err})
    """
    if interp_methods is None:
        interp_methods = VALIDATION_INTERPOLATION_METHODS

    # normalize input to list
    if isinstance(trip_list, (pl.DataFrame,)):
        trip_list = [trip_list]

    def _eval_single(trip_df, interpolator):
        t_start = trip_df["t_event"][0]
        t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
        s_vals = trip_df["s_m"].to_numpy()
        ts = np.linspace(t_sec[0], t_sec[-1], 500)

        interp64 = _build_validation_interpolator(
            t_sec,
            s_vals,
            interpolator,
            trip_df["v_clamp"].to_numpy() if "v_clamp" in trip_df.columns else None,
        )

        s64 = interp64(ts)
        v64 = (
            interp64.derivative(1)(ts)
            if hasattr(interp64, "derivative")
            else np.gradient(s64, ts)
        )
        a64 = (
            interp64.derivative(2)(ts)
            if hasattr(interp64, "derivative")
            else np.gradient(v64, ts)
        )

        out = {"float64": {"v": v64, "a": a64}}

        for dtype in [np.float32, np.float16]:
            dtype_name = np.dtype(dtype).name
            try:
                s_cast = s_vals.astype(dtype)
                v_clamp_cast = (
                    trip_df["v_clamp"].to_numpy().astype(dtype)
                    if "v_clamp" in trip_df.columns
                    else None
                )
                interp = _build_validation_interpolator(
                    t_sec.astype(dtype), s_cast, interpolator, v_clamps=v_clamp_cast
                )
                s_t = interp(ts)
                v_t = (
                    interp.derivative(1)(ts)
                    if hasattr(interp, "derivative")
                    else np.gradient(s_t, ts)
                )
                a_t = (
                    interp.derivative(2)(ts)
                    if hasattr(interp, "derivative")
                    else np.gradient(v_t, ts)
                )

                v_err = np.nanmean(np.abs(v_t - v64))
                a_err = np.nanmean(np.abs(a_t - a64))
                out[dtype_name] = {
                    "v_err_mean": float(v_err),
                    "a_err_mean": float(a_err),
                }
            except Exception as e:
                out[dtype_name] = {"error": str(e)}

        return out

    # Accumulate metrics per method and dtype across trips
    summary = {}
    for method in interp_methods:
        metrics = {
            "float32": {"v": [], "a": []},
            "float16": {"v": [], "a": []},
        }
        for trip_df in trip_list:
            try:
                res = _eval_single(trip_df, method)
            except Exception:
                continue
            for dtype in ["float32", "float16"]:
                if dtype in res and "v_err_mean" in res[dtype]:
                    metrics[dtype]["v"].append(res[dtype]["v_err_mean"])
                    metrics[dtype]["a"].append(res[dtype]["a_err_mean"])

        # compute mean/std across trips
        summary[method] = {}
        for dtype in ["float32", "float16"]:
            v_arr = (
                np.array(metrics[dtype]["v"]) if metrics[dtype]["v"] else np.array([])
            )
            a_arr = (
                np.array(metrics[dtype]["a"]) if metrics[dtype]["a"] else np.array([])
            )
            if v_arr.size:
                summary[method][dtype] = {
                    "mean_v_err": float(v_arr.mean()),
                    "std_v_err": float(v_arr.std()),
                    "mean_a_err": float(a_arr.mean()) if a_arr.size else None,
                    "std_a_err": float(a_arr.std()) if a_arr.size else None,
                    "n": int(v_arr.size),
                }
            else:
                summary[method][dtype] = {
                    "mean_v_err": None,
                    "std_v_err": None,
                    "mean_a_err": None,
                    "std_a_err": None,
                    "n": 0,
                }

    if plot:
        methods = list(summary.keys())
        x = np.arange(len(methods))
        w = 0.35

        f32_means = [summary[m]["float32"]["mean_v_err"] or 0.0 for m in methods]
        f16_means = [summary[m]["float16"]["mean_v_err"] or 0.0 for m in methods]
        f32_err = [summary[m]["float32"]["std_v_err"] or 0.0 for m in methods]
        f16_err = [summary[m]["float16"]["std_v_err"] or 0.0 for m in methods]

        # Show both linear and log scales because float32 error is often orders of magnitude smaller.
        fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

        axs[0].bar(x - w / 2, f32_means, w, yerr=f32_err, label="float32")
        axs[0].bar(x + w / 2, f16_means, w, yerr=f16_err, label="float16")
        axs[0].set_xticks(x)
        axs[0].set_xticklabels(methods, rotation=45)
        axs[0].set_ylabel("Mean |delta velocity| against float64 (m/s)")
        axs[0].set_title("Precision Degradation (Linear Scale)")
        axs[0].legend()
        axs[0].grid(axis="y", alpha=0.3)

        eps = 1e-8
        f32_log = [max(v, eps) for v in f32_means]
        f16_log = [max(v, eps) for v in f16_means]
        axs[1].bar(x - w / 2, f32_log, w, yerr=f32_err, label="float32")
        axs[1].bar(x + w / 2, f16_log, w, yerr=f16_err, label="float16")
        axs[1].set_yscale("log")
        axs[1].set_xticks(x)
        axs[1].set_xticklabels(methods, rotation=45)
        axs[1].set_ylabel("Mean |delta velocity| against float64 (m/s), log scale")
        axs[1].set_title("Precision Degradation (Log Scale)")
        axs[1].legend()
        axs[1].grid(axis="y", alpha=0.3, which="both")

        plt.tight_layout()
        plt.show()

        # Print table
        print(
            "Precision degradation summary (mean |delta v| and |delta a| across trips):"
        )
        for m in methods:
            print(f" - {m}:")
            for dtype in ["float32", "float16"]:
                s = summary[m][dtype]
                if s["n"]:
                    print(
                        f"    {dtype}: mean_v_err={s['mean_v_err']:.6f}, std_v={s['std_v_err']:.6f}, "
                        f"mean_a_err={s['mean_a_err']:.6f}, std_a={s['std_a_err']:.6f}, n={s['n']}"
                    )
                else:
                    print(f"    {dtype}: no valid runs")

    return summary

In [ ]:
def noise_injection_analysis(
    trip_list,
    sigma_t=0.2,
    sigma_s=0.5,
    trials=10,
    interp_methods=None,
    plot=True,
):
    """Inject Gaussian noise into multiple trips and interpolation methods, summarize velocity variance.

    Args:
        trip_list: list of per-trip DataFrames (or a single trip_df).
        sigma_t, sigma_s: noise stddevs for time and space.
        trials: number of noisy trials per trip.
        interp_methods: list of interpolation methods to test. If None uses VALIDATION_INTERPOLATION_METHODS.
        plot: whether to produce summary plots.

    Returns:
        summary: dict[method] -> {trials_total, mean_v_mse, std_v_mse}
    """
    if interp_methods is None:
        interp_methods = VALIDATION_INTERPOLATION_METHODS

    if isinstance(trip_list, (pl.DataFrame,)):
        trip_list = [trip_list]

    def _single_noisy_mse(trip_df, method):
        t_start = trip_df["t_event"][0]
        t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
        s_vals = trip_df["s_m"].to_numpy()
        ts_eval = np.linspace(t_sec[0], t_sec[-1], 500)

        baseline = _build_validation_interpolator(
            t_sec,
            s_vals,
            method,
            trip_df["v_clamp"].to_numpy() if "v_clamp" in trip_df.columns else None,
        )
        v_baseline = (
            baseline.derivative(1)(ts_eval)
            if hasattr(baseline, "derivative")
            else np.gradient(baseline(ts_eval), ts_eval)
        )

        v_vars = []
        for _ in range(trials):
            t_noisy = t_sec + np.random.normal(0.0, sigma_t, size=t_sec.shape)
            s_noisy = s_vals + np.random.normal(0.0, sigma_s, size=s_vals.shape)
            try:
                interp = _build_validation_interpolator(t_noisy, s_noisy, method)
                v_n = (
                    interp.derivative(1)(ts_eval)
                    if hasattr(interp, "derivative")
                    else np.gradient(interp(ts_eval), ts_eval)
                )
                v_vars.append(np.nanmean((v_n - v_baseline) ** 2))
            except Exception:
                continue

        return v_vars

    summary = {}
    for method in interp_methods:
        all_vars = []
        for trip_df in trip_list:
            try:
                vvars = _single_noisy_mse(trip_df, method)
            except Exception:
                vvars = []
            all_vars.extend(vvars)

        if all_vars:
            summary[method] = {
                "trials": len(all_vars),
                "v_mse_mean": float(np.mean(all_vars)),
                "v_mse_std": float(np.std(all_vars)),
            }
        else:
            summary[method] = {"trials": 0, "v_mse_mean": None, "v_mse_std": None}

    if plot:
        methods = list(summary.keys())
        means = [summary[m]["v_mse_mean"] or 0.0 for m in methods]
        errs = [summary[m]["v_mse_std"] or 0.0 for m in methods]

        # Use both linear and log scales because cubic methods can dominate the axis.
        fig, axs = plt.subplots(1, 2, figsize=(16, 5), sharex=True)

        axs[0].bar(methods, means, yerr=errs, color="steelblue", alpha=0.9)
        axs[0].set_ylabel("Mean velocity MSE (vs baseline)")
        axs[0].set_title(
            f"Noise Injection Analysis (Linear, sigma_t={sigma_t}, sigma_s={sigma_s})"
        )
        axs[0].grid(axis="y", alpha=0.3)

        eps = 1e-10
        means_log = [max(v, eps) for v in means]
        axs[1].bar(methods, means_log, yerr=errs, color="seagreen", alpha=0.9)
        axs[1].set_yscale("log")
        axs[1].set_ylabel("Mean velocity MSE (log scale)")
        axs[1].set_title(
            f"Noise Injection Analysis (Log, sigma_t={sigma_t}, sigma_s={sigma_s})"
        )
        axs[1].grid(axis="y", alpha=0.3, which="both")

        for ax in axs:
            ax.tick_params(axis="x", rotation=30)

        plt.tight_layout()
        plt.show()

        print("Noise injection summary:")
        for m in methods:
            s = summary[m]
            if s["trials"]:
                print(
                    f" - {m}: trials={s['trials']}, mean_mse={s['v_mse_mean']:.6f}, std={s['v_mse_std']:.6f}"
                )
            else:
                print(f" - {m}: no valid trials")

    return summary

In [ ]:
# Build a consistent trip sample for all robustness tests
trip_ids_eval = (
    df_final_knots.select("trip_id")
    .unique()
    .sample(min(50, df_final_knots["trip_id"].n_unique()), seed=42)
    .get_column("trip_id")
    .to_list()
)

trip_list_eval = [
    df_final_knots.filter(pl.col("trip_id") == tid).sort("t_event")
    for tid in trip_ids_eval
]

methods_eval = ["pchip", "bpoly", "cubic_clamped", "cubic_natural", "linear"]

print(f"Running precision degradation on {len(trip_list_eval)} trips...")
precision_summary = evaluate_precision_degradation(
    trip_list_eval,
    interp_methods=methods_eval,
    plot=True,
)

print("\nRunning noise injection analysis on same trip sample...")
noise_summary = noise_injection_analysis(
    trip_list_eval,
    sigma_t=0.2,
    sigma_s=0.5,
    trials=20,
    interp_methods=methods_eval,
    plot=True,
)

precision_summary, noise_summary

In [ ]:
def evaluate_interp_monotonicity(trip_df, interp_method="cubic"):
    """
    Checks if a selected interpolation method fitted to the trip data ever results in negative velocity.
    """
    # 1. Extract your time and distance arrays
    t_start = trip_df["t_event"][0]
    t_sec = np.array([(t - t_start).total_seconds() for t in trip_df["t_event"]])
    s_vals = trip_df["s_m"].to_numpy()

    interp = _build_validation_interpolator(t_sec, s_vals, interp_method)
    # 2. Fit the standard Cubic Spline
    # cs = CubicSpline(t_sec, s_vals)

    # 3. Get the first derivative function (Velocity)
    v_func = interp.derivative(nu=1)

    # 4. Create a dense evaluation grid (e.g., check the velocity every 1 second)
    t_dense = np.linspace(t_sec[0], t_sec[-1], num=int(t_sec[-1] - t_sec[0]) + 1)

    # 5. Calculate velocities across the whole trip
    velocities = v_func(t_dense)

    # TODO: put in paper that we couldn't use 0 due to numerical noise causing tiny negative values even when the curve is basically monotone increasing, so we had to use a small tolerance threshold to avoid false positives.
    # 6. Analyze the results
    negative_velocities = (
        velocities < -1e-3
    )  # Allow a tiny tolerance for numerical noise
    is_non_monotonic = np.any(negative_velocities)
    percent_backward = np.sum(negative_velocities) / len(velocities) * 100

    return is_non_monotonic, percent_backward

In [ ]:
trip_ids = df_final_knots["trip_id"].unique().to_list()
trips_with_reversals = 0
total_reversal_time = []

for tid in trip_ids:
    trip_df = df_final_knots.filter(pl.col("trip_id") == tid).sort("t_event")

    # Skip trips with too few points
    if trip_df.height < 4:
        continue

    is_backward, percent_back = evaluate_interp_monotonicity(trip_df)

    if is_backward:
        trips_with_reversals += 1
        total_reversal_time.append(percent_back)

print(f"Total trips analyzed: {len(trip_ids)}")
print(
    f"Trips with negative velocity: {trips_with_reversals} ({(trips_with_reversals / len(trip_ids)) * 100:.1f}%)"
)

if total_reversal_time:
    print(
        f"Average time spent moving backward (on flawed trips): {np.mean(total_reversal_time):.1f}%"
    )

In [ ]:
trip_ids = df_final_knots["trip_id"].unique().to_list()
trips_with_reversals = 0
total_reversal_time = []

for tid in trip_ids:
    trip_df = df_final_knots.filter(pl.col("trip_id") == tid).sort("t_event")

    # Skip trips with too few points
    if trip_df.height < 4:
        continue

    is_backward, percent_back = evaluate_interp_monotonicity(
        trip_df, interp_method="pchip"
    )

    if is_backward:
        trips_with_reversals += 1
        total_reversal_time.append(percent_back)

print(f"Total trips analyzed: {len(trip_ids)}")
print(
    f"Trips with negative velocity: {trips_with_reversals} ({(trips_with_reversals / len(trip_ids)) * 100:.1f}%)"
)

if total_reversal_time:
    print(
        f"Average time spent moving backward (on flawed trips): {np.mean(total_reversal_time):.1f}%"
    )